# Coleta dos dados

Baixa as despesas dos 12 municípios da AMREC, de 2023 a 2026, para `dados/brutos/amrec/`.

- **Betha** (10 municípios): coleta automática pela API pública do portal de transparência, registro a registro.
- **Atende.net** (Treviso e Morro da Fumaça): o portal tem verificação anti-robô, então os CSVs são exportados
  à mão. Este notebook só confere se os arquivos estão no lugar.

A coleta do Betha leva horas, porque o servidor entrega poucos registros por segundo. Ela é retomável: se for
interrompida, basta rodar de novo, e só o que falta é baixado. O progresso detalhado vai para
`dados/brutos/amrec/coleta.log`; aqui aparecem só o resultado de cada consulta e a conferência final.

Os dados brutos têm CPF e CNPJ de credores e ficam fora do Git (`dados/` está no `.gitignore`).

In [1]:
import base64
import ctypes
import datetime
import http.client
import json
import logging
import sys
import threading
import time
import unicodedata
import urllib.error
import urllib.parse
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import pandas as pd

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

MUNICIPIOS_BETHA = {  # IBGE: (pasta, hash do portal da prefeitura)
    "4220000": ("balneario_rincao", "RKnJOJYwtLqFoxpLQvqz1w=="),
    "4204251": ("cocal_do_sul", "qtu0z4J_77QAd8cuvwU9yw=="),
    "4204608": ("criciuma", "n4W91vnHptoBkiHKAxioOA=="),
    "4205456": ("forquilhinha", "YFfgynIaJpiC538_iw4kRQ=="),
    "4207007": ("icara", "h0C3qPTly07TR2VXjKm9fg=="),
    "4209607": ("lauro_muller", "UJfapwFZe_btE-Ps-7gcKw=="),
    "4211603": ("nova_veneza", "9W-1qcRgfIHLV6eWLQzpCQ=="),
    "4211702": ("orleans", "mJRAZlfGu2RI9KNBWj49aw=="),
    "4217600": ("sideropolis", "c4BnkBx8phYUcyBpfTlVxg=="),
    "4219002": ("urussanga", "sbLYyLl4bImD4bM6JRvsjg=="),
}
MUNICIPIOS_IPM = {"4218350": "treviso", "4211207": "morro_da_fumaca"}

DESTINO = RAIZ / "dados" / "brutos" / "amrec"
COLETAR = list(MUNICIPIOS_BETHA)
ANOS = list(range(2023, 2027))
PASSADAS = 2
TELA_LIGADA = False

## 1. Cliente da API do Betha

O portal usa um token anônimo, o mesmo que o navegador recebe ao abrir a página. Cada requisição informa o
portal da prefeitura no cabeçalho `App-Context`. Em queda de rede, token expirado (401), excesso de requisições
(429) ou erro do servidor (5xx), a chamada é repetida com espera crescente; outros erros são devolvidos na hora.

In [2]:
API = "https://api.transparencia.betha.cloud/transparencia"
OAUTH = "https://plataforma-oauth.betha.cloud/auth/oauth2/authorize"
CLIENT_ID = "91a97459-f1d8-4b29-b5fa-2e51d1692623"
REDIRECT = "https://transparencia.betha.cloud/auth-callback.html"
USER_AGENT = "Mozilla/5.0 (trabalho academico; coleta de dados publicos)"
TENTATIVAS = 6
ERROS_DE_REDE = (urllib.error.URLError, http.client.HTTPException, ConnectionError, TimeoutError)


class SemRedirecionar(urllib.request.HTTPRedirectHandler):
    def redirect_request(self, *args, **kwargs):
        return None


class Token:
    """Token anônimo compartilhado entre os portais, renovado antes de expirar."""

    def __init__(self):
        self._valor = None
        self._expira = 0.0
        self._trava = threading.Lock()

    def obter(self, renovar=False):
        with self._trava:
            if renovar or self._valor is None or time.time() > self._expira - 120:
                self._valor, self._expira = self._novo()
            return self._valor

    @staticmethod
    def _novo():
        parametros = urllib.parse.urlencode({
            "client_id": CLIENT_ID,
            "response_type": "token",
            "redirect_uri": REDIRECT,
            "scope": "transparencia.public",
            "access_mode": "anonymous",
        })
        pedido = urllib.request.Request(f"{OAUTH}?{parametros}", headers={"User-Agent": USER_AGENT})
        try:
            urllib.request.build_opener(SemRedirecionar).open(pedido, timeout=60)
        except urllib.error.HTTPError as resposta:
            destino = resposta.headers.get("Location", "")
        else:
            raise RuntimeError("o OAuth do Betha não redirecionou")
        fragmento = dict(urllib.parse.parse_qsl(destino.partition("#")[2]))
        return fragmento["access_token"], time.time() + int(fragmento.get("expires_in", 600))


token = Token()


class Portal:
    def __init__(self, hash_portal):
        contexto = json.dumps({"portal": hash_portal}, separators=(",", ":"))
        self._contexto = base64.b64encode(contexto.encode()).decode()

    def menu(self):
        return self._requisitar("GET", "/api/menu")

    def filtros(self, consulta):
        return self._requisitar("GET", f"/api/consulta/{consulta}/filtros")

    def buscar(self, consulta, filtro, limite):
        caminho = f"/api/busca-textual/{consulta}?offset=0&limit={limite}&hiperlink=false"
        return self._requisitar("POST", caminho, filtro, timeout=900)

    def detalhe(self, consulta, id_registro):
        return self._requisitar("GET", f"/api/busca/{consulta}/detalhes/{id_registro}")

    def _requisitar(self, metodo, caminho, corpo=None, timeout=120):
        dados = None if corpo is None else json.dumps(corpo).encode()
        renovar = False
        for tentativa in range(TENTATIVAS):
            pedido = urllib.request.Request(f"{API}{caminho}", data=dados, method=metodo,
                                            headers=self._cabecalhos(renovar))
            try:
                with urllib.request.urlopen(pedido, timeout=timeout) as resposta:
                    return json.loads(resposta.read())
            except urllib.error.HTTPError as erro:
                if erro.code < 500 and erro.code not in (401, 429):
                    raise
                renovar, ultimo = erro.code == 401, erro
            except (*ERROS_DE_REDE, json.JSONDecodeError) as erro:
                renovar, ultimo = False, erro
            time.sleep(5 * (tentativa + 1))
        raise RuntimeError(f"{metodo} {caminho}: {ultimo}")

    def _cabecalhos(self, renovar):
        return {
            "Authorization": f"Bearer {token.obter(renovar)}",
            "App-Context": self._contexto,
            "User-Agent": USER_AGENT,
            "Accept": "application/json",
            "Content-Type": "application/json",
            "Origin": "https://transparencia.betha.cloud",
            "Referer": "https://transparencia.betha.cloud/",
        }

## 2. Coleta de uma consulta

Cada prefeitura tem IDs próprios para as consultas, então elas são localizadas pelo título no menu do portal.
Para cada consulta:

1. lista os IDs de cada ano e confere se a quantidade bate com o total informado pelo portal;
2. baixa o detalhe de cada registro ainda não gravado, com 2 conexões por município;
3. grava um registro por linha no `.jsonl`, com `_id`, `_ano`, `_ibge` e `_consulta`;
4. grava em `<consulta>_ids.json` a lista de IDs que o portal publica hoje; o tratamento usa só esses registros,
   descartando os que foram retirados do portal depois de baixados;
5. devolve o total do portal, o total baixado e o status (`OK` só quando os dois são iguais).

Ao reabrir um arquivo, uma última linha cortada por queda no meio da gravação e registros repetidos são removidos.

In [3]:
CONSULTAS = {  # título no menu do portal: arquivo
    "EXECUCAO DETALHADA DE DESPESAS": "execucao_detalhada",
    "DIARIAS": "diarias",
    "RECEITAS E DESPESAS EXTRAORCAMENTARIAS": "extraorcamentarias",
    "DESPESAS ORCAMENTARIAS": "despesas_orcamentarias",
}
CONEXOES_POR_MUNICIPIO = 2
PAUSA = 0.3


def normalizar(texto):
    sem_acento = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode()
    return " ".join(sem_acento.upper().split())


def localizar_consultas(portal):
    encontradas = {}

    def percorrer(itens):
        for item in itens:
            for consulta in item.get("consultas") or []:
                encontradas.setdefault(normalizar(consulta["titulo"]), consulta["id"])
            percorrer(item.get("subMenus") or [])

    percorrer(portal.menu())
    return {arquivo: encontradas.get(titulo) for titulo, arquivo in CONSULTAS.items()}


def listar_ids(portal, consulta, anos):
    chave = next((f["key"] for f in portal.filtros(consulta) if f["key"] in ("ano", "anoExercicio")), None)
    if chave is None:
        raise RuntimeError("a consulta não tem filtro de ano")
    ids = {}
    for ano in anos:
        filtro = {chave: [ano]}
        total = portal.buscar(consulta, filtro, 1)["totalHits"]
        registros = portal.buscar(consulta, filtro, total)["hits"] if total else []
        ids[ano] = list(dict.fromkeys(r["id"] for r in registros))
        if len(ids[ano]) != total:
            raise RuntimeError(f"{ano}: o portal informou {total} registros e listou {len(ids[ano])}")
    return ids


def ler_baixados(arquivo):
    if not arquivo.exists():
        return set()
    texto = arquivo.read_text(encoding="utf-8")
    ids, validas = set(), []
    for linha in texto.splitlines():
        try:
            id_registro = json.loads(linha)["_id"]
        except (json.JSONDecodeError, KeyError, TypeError):
            continue
        if id_registro not in ids:
            ids.add(id_registro)
            validas.append(linha + "\n")
    limpo = "".join(validas)
    if limpo != texto:
        arquivo.write_text(limpo, encoding="utf-8")
    return ids


def coletar_consulta(portal, consulta, arquivo, ibge, log):
    ids = listar_ids(portal, consulta, ANOS)
    ano_do_id = {}
    for ano, lista in ids.items():
        for id_registro in lista:
            ano_do_id.setdefault(id_registro, ano)
    arquivo.with_name(f"{arquivo.stem}_ids.json").write_text(json.dumps(ids), encoding="utf-8")
    baixados = ler_baixados(arquivo) & ano_do_id.keys()
    pendentes = [i for i in ano_do_id if i not in baixados]
    log.info("%s: %d no portal, %d já baixados, %d a baixar",
             arquivo.stem, len(ano_do_id), len(baixados), len(pendentes))

    falhas = []
    trava = threading.Lock()
    with arquivo.open("a", encoding="utf-8") as saida:

        def baixar(id_registro):
            ano = ano_do_id[id_registro]
            try:
                detalhe = portal.detalhe(consulta, id_registro)
                linha = json.dumps({"_id": id_registro, "_ano": ano, "_ibge": ibge, "_consulta": consulta,
                                    **detalhe}, ensure_ascii=False)
            except Exception as erro:
                with trava:
                    falhas.append(f"{id_registro}\t{ano}\t{erro}\n")
            else:
                with trava:
                    saida.write(linha + "\n")
                    saida.flush()
                    baixados.add(id_registro)
                    if len(baixados) % 1000 == 0:
                        log.debug("%s: %d/%d", arquivo.stem, len(baixados), len(ano_do_id))
            time.sleep(PAUSA)

        with ThreadPoolExecutor(CONEXOES_POR_MUNICIPIO) as executor:
            list(executor.map(baixar, pendentes))

    arquivo_falhas = arquivo.with_name(f"{arquivo.stem}_falhas.log")
    if falhas:
        arquivo_falhas.write_text("".join(falhas), encoding="utf-8")
    else:
        arquivo_falhas.unlink(missing_ok=True)
    return {
        "consulta": consulta,
        "arquivo": arquivo.name,
        "total_portal": len(ano_do_id),
        "por_ano": {ano: len(lista) for ano, lista in ids.items()},
        "baixados": len(baixados),
        "falhas": len(falhas),
        "status": "OK" if len(baixados) == len(ano_do_id) else "INCOMPLETO",
    }

## 3. Coleta de um município

Percorre as quatro consultas e atualiza o `manifesto.json` da pasta do município a cada consulta concluída.
Uma consulta que o portal não publica fica como `AUSENTE`; um erro na listagem fica como `ERRO` e é refeito na
passada seguinte.

In [4]:
def agora():
    return datetime.datetime.now().isoformat(timespec="seconds")


def salvar_manifesto(pasta, manifesto):
    (pasta / "manifesto.json").write_text(json.dumps(manifesto, ensure_ascii=False, indent=1), encoding="utf-8")


def coletar_municipio(ibge):
    nome, hash_portal = MUNICIPIOS_BETHA[ibge]
    pasta = DESTINO / f"{ibge}_{nome}"
    pasta.mkdir(parents=True, exist_ok=True)
    log = logging.getLogger(f"coleta.{nome}")
    portal = Portal(hash_portal)
    manifesto = {"ibge": ibge, "municipio": nome, "anos": ANOS, "inicio": agora(), "consultas": {}}

    try:
        consultas = localizar_consultas(portal)
    except Exception as erro:
        log.error("não foi possível ler o menu do portal: %s", erro)
        manifesto["erro"] = str(erro)
        salvar_manifesto(pasta, manifesto)
        return False

    for arquivo, consulta in consultas.items():
        if consulta is None:
            resultado = {"status": "AUSENTE"}
        else:
            try:
                resultado = coletar_consulta(portal, consulta, pasta / f"{arquivo}.jsonl", ibge, log)
            except Exception as erro:
                resultado = {"consulta": consulta, "status": "ERRO", "erro": str(erro)}
        log.info("%s: %s", arquivo, resultado["status"])
        manifesto["consultas"][arquivo] = resultado
        salvar_manifesto(pasta, manifesto)

    manifesto["fim"] = agora()
    salvar_manifesto(pasta, manifesto)
    return all(r["status"] in ("OK", "AUSENTE") for r in manifesto["consultas"].values())

## 4. Execução

Os municípios rodam ao mesmo tempo. Ao fim de cada passada, os que tiveram alguma consulta incompleta entram na
passada seguinte. Enquanto a célula roda, o Windows é impedido de suspender (`TELA_LIGADA = True` também mantém a
tela acesa); o pedido acaba quando o kernel é encerrado.

In [5]:
def configurar_log():
    log = logging.getLogger("coleta")
    log.setLevel(logging.DEBUG)
    for antigo in list(log.handlers):
        antigo.close()
        log.removeHandler(antigo)
    formato = logging.Formatter("%(asctime)s [%(name)s] %(message)s", "%H:%M:%S")
    arquivo = logging.FileHandler(DESTINO / "coleta.log", encoding="utf-8")
    arquivo.setFormatter(formato)
    tela = logging.StreamHandler(sys.stdout)
    tela.setLevel(logging.INFO)
    tela.setFormatter(formato)
    log.addHandler(arquivo)
    log.addHandler(tela)
    return log


def impedir_suspensao():
    if sys.platform == "win32":
        continuo, sistema, tela = 0x80000000, 0x00000001, 0x00000002
        ctypes.windll.kernel32.SetThreadExecutionState(continuo | sistema | (tela if TELA_LIGADA else 0))


DESTINO.mkdir(parents=True, exist_ok=True)
log = configurar_log()
impedir_suspensao()

pendentes = list(COLETAR)
for passada in range(1, PASSADAS + 1):
    log.info("passada %d de %d: %d municípios", passada, PASSADAS, len(pendentes))
    with ThreadPoolExecutor(len(pendentes)) as executor:
        concluidos = list(executor.map(coletar_municipio, pendentes))
    pendentes = [ibge for ibge, ok in zip(pendentes, concluidos) if not ok]
    if not pendentes:
        break
    if passada < PASSADAS:
        time.sleep(60)

log.info("coleta concluída" if not pendentes else f"incompletos: {', '.join(pendentes)}")

15:58:30 [coleta] passada 1 de 2: 10 municípios


15:58:55 [coleta.nova_veneza] execucao_detalhada: 14904 no portal, 14903 já baixados, 1 a baixar


15:58:59 [coleta.nova_veneza] execucao_detalhada: OK


15:59:10 [coleta.balneario_rincao] execucao_detalhada: 29709 no portal, 29707 já baixados, 2 a baixar


15:59:12 [coleta.balneario_rincao] execucao_detalhada: OK


15:59:12 [coleta.nova_veneza] diarias: 56 no portal, 56 já baixados, 0 a baixar


... (68 outputs left out)

16:10:07 [coleta.criciuma] extraorcamentarias: OK


16:10:17 [coleta.criciuma] despesas_orcamentarias: 1518 no portal, 1518 já baixados, 0 a baixar


16:10:17 [coleta.criciuma] despesas_orcamentarias: OK


16:10:17 [coleta] coleta concluída


## 5. Conferência do Betha

Total informado pelo portal contra total gravado, por município e consulta, lido dos manifestos.

In [6]:
linhas = []
for ibge, (nome, _) in MUNICIPIOS_BETHA.items():
    caminho = DESTINO / f"{ibge}_{nome}" / "manifesto.json"
    if not caminho.exists():
        continue
    manifesto = json.loads(caminho.read_text(encoding="utf-8"))
    for consulta, resultado in manifesto["consultas"].items():
        linhas.append({
            "municipio": nome,
            "consulta": consulta,
            "portal": resultado.get("total_portal"),
            "baixados": resultado.get("baixados"),
            "status": resultado["status"],
        })

conferencia = pd.DataFrame(linhas)
assert conferencia["status"].isin(["OK", "AUSENTE"]).all(), "há consultas incompletas: rode a coleta de novo"
conferencia.pivot(index="municipio", columns="consulta", values="baixados").astype("Int64")

consulta,despesas_orcamentarias,diarias,execucao_detalhada,extraorcamentarias
municipio,,,,
balneario_rincao,621,2808,29709,<NA>
cocal_do_sul,754,1181,46351,1159
criciuma,1518,11447,62747,1574
forquilhinha,873,909,43076,2119
icara,1389,3575,59683,2618
lauro_muller,629,1207,31396,<NA>
nova_veneza,515,56,14904,701
orleans,601,320,39027,1029
sideropolis,644,181,26000,893


## 6. Atende.net (Treviso e Morro da Fumaça)

Exportação manual no portal de cada município (`https://<municipio>.atende.net/transparencia/grupo/despesas`),
em CSV, um arquivo por ano de 2023 a 2026, para cada uma das consultas:

| Consulta no portal | Nome do arquivo |
| --- | --- |
| Empenhos | `empenho_emitido_<ano>.csv` |
| Liquidações | `liquidacao_<ano>.csv` |
| Pagamentos | `pagamentos_<ano>.csv` |
| Diárias | `diarias_<ano>.csv` |
| Orçamento da despesa | `orcamento_despesa_<ano>.csv` |

O arquivo de orçamento não traz o ano; o nome do arquivo é a única referência. A célula abaixo confere se os 40
arquivos estão no lugar.

In [7]:
EXPORTACOES = ["empenho_emitido", "liquidacao", "pagamentos", "diarias", "orcamento_despesa"]

arquivos_ipm = pd.DataFrame([
    {"municipio": nome, "consulta": consulta, "ano": ano,
     "kb": (caminho.stat().st_size // 1024) if caminho.exists() else None}
    for ibge, nome in MUNICIPIOS_IPM.items()
    for consulta in EXPORTACOES
    for ano in ANOS
    for caminho in [DESTINO / f"{ibge}_{nome}" / f"{consulta}_{ano}.csv"]
])
faltando = arquivos_ipm[arquivos_ipm["kb"].isna()]
assert faltando.empty, f"arquivos faltando:\n{faltando}"
arquivos_ipm.astype({"kb": int}).pivot(index=["municipio", "consulta"], columns="ano", values="kb")

ano                                 2023   2024   2025   2026
municipio       consulta                                     
morro_da_fumaca diarias              365    406    535    384
                empenho_emitido     4524   4858   4820   4082
                liquidacao          2047   2184   2235   1867
                orcamento_despesa     11     11     12     12
                pagamentos         12619  15309  15124  13857
treviso         diarias                6     14     21     19
                empenho_emitido     2300   2582   2616   2528
                liquidacao           827    859    920    869
                orcamento_despesa      6      6      6      7
                pagamentos          5067   6338   6460   5755